In [12]:
# Importa bibliotecas necessárias
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout

In [13]:
# Definindo os caminho da nova pasta no PC pessoal
# O caminho usa '../../' pois o notebook está em scripts/Redução de Dimensionalidade/
caminho_pasta_tratado = '../../dataset tratado/lycos-cicids2017/'

nome_dados_treinamento = 'lycos_cicids2017_treinamento.csv'
nome_dados_teste = 'lycos_cicids2017_teste.csv'

nome_dados_treinamento_reduzidos = 'Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_balanceados.csv'
nome_dados_teste_reduzidos = 'Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_balanceados.csv'

In [14]:
# 1. Carregando o input a partir dos CSVs de dados JÁ TRATADOS
print("Carregando os datasets tratados em CSV...")
df_treino = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")

display(df_treino.head())

Carregando os datasets tratados em CSV...
Dados carregados! Treino: (1286248, 80) | Teste: (551250, 80)


,src_port,dst_port,ip_prot,timestamp,flow_duration,down_up_ratio,pkt_len_max,pkt_len_min,pkt_len_mean,pkt_len_var,...,bwd_bulk_bytes_mean,bwd_bulk_pkt_mean,bwd_bulk_rate_mean,fwd_subflow_bytes_mean,fwd_subflow_pkt_mean,bwd_subflow_bytes_mean,bwd_subflow_pkt_mean,fwd_tcp_init_win_bytes,bwd_tcp_init_win_bytes,Label
0,0.951110,0.000809,0.122137,0.493151,0.000504,0.123909,0.003948,0.031768,0.030108,0.000050,...,0.000000,0.000000,0.000000,0.000026,0.000005,1.495151e-07,0.000003,0.000000,0.000000,benign
1,0.766598,0.006760,0.038168,0.235429,0.025492,0.041303,0.001249,0.000000,0.003345,0.000009,...,0.000000,0.000000,0.000000,0.000009,0.000007,0.000000e+00,0.000002,0.003922,0.021423,benign
2,0.814466,0.006760,0.038168,0.512447,0.979619,0.111518,0.057131,0.000000,0.066731,0.004397,...,0.000004,0.000069,0.000009,0.000228,0.000030,2.463439e-06,0.000021,0.445572,0.647110,benign
3,0.074983,0.000809,0.122137,0.240826,0.000002,0.123909,0.010475,0.022790,0.061262,0.000639,...,0.000000,0.000000,0.000000,0.000037,0.000009,7.933453e-07,0.000007,0.000000,0.000000,benign
4,0.847898,0.006760,0.038168,0.235855,0.070290,0.149890,0.175020,0.000000,0.392759,0.041693,...,0.000387,0.000566,0.000418,0.000322,0.000094,6.456254e-05,0.000086,0.445572,0.176773,benign


In [15]:
# 2. Removendo características identificadoras/enviesadas antes da seleção por MDI
# No dataset tratado atual, o campo identificador/enviesado ainda presente é a porta de destino.
# A coluna duplicada de Fwd Header Length também é removida para reduzir redundância.
nomes_enviesados = {
    'Flow ID',
    'Source IP',
    'Source Port',
    'Destination IP',
    'Destination Port',
    'Protocol',
    'Timestamp',
    'timestamp',
    'Fwd Header Length.1'
}

colunas_para_remover = []
fwd_header_length_vistos = 0

for coluna in df_treino.columns:
    nome_normalizado = coluna.strip().lower()

    if coluna in nomes_enviesados:
        colunas_para_remover.append(coluna)

    if nome_normalizado == 'fwd header length':
        fwd_header_length_vistos += 1
        if fwd_header_length_vistos > 1:
            colunas_para_remover.append(coluna)

    if nome_normalizado.startswith('fwd header length.'):
        colunas_para_remover.append(coluna)

colunas_para_remover = [coluna for coluna in dict.fromkeys(colunas_para_remover) if coluna != 'Label']
colunas_treino_existentes = [coluna for coluna in colunas_para_remover if coluna in df_treino.columns]
colunas_teste_existentes = [coluna for coluna in colunas_para_remover if coluna in df_teste.columns]

df_treino = df_treino.drop(columns=colunas_treino_existentes)
df_teste = df_teste.drop(columns=colunas_teste_existentes)

print('Características removidas para mitigar overfitting:', colunas_treino_existentes)
print(f'Dimensões após remoção - Treino: {df_treino.shape} | Teste: {df_teste.shape}')

# 3. Separando as características (X) do gabarito/rótulo (y)
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

Características removidas para mitigar overfitting: ['timestamp']
Dimensões após remoção - Treino: (1286248, 79) | Teste: (551250, 79)


In [16]:
# 4. Aplicando Seleção de Características MDI (Mean Decrease in Impurity) com Random Forest Balanceado
print("\nIniciando o treinamento do Random Forest para extração de features (MDI)...")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, class_weight='balanced')

inicio_fs = time.time()
rf_fs.fit(X_treino, y_treino)
fim_fs = time.time()

print(f"Treinamento RF concluído! Tempo extração: {(fim_fs - inicio_fs):.4f} segundos.")

importancias = rf_fs.feature_importances_
features = X_treino.columns

df_importancias = pd.DataFrame({'Feature': features, 'Importancia': importancias})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)

print("\nQuantidade total de features:", len(features))

# Selecionando as 51 features mais importantes (conforme sugerido na literatura por Neto e Gomes)
top_51_features = df_importancias.head(51)['Feature'].tolist()

quantidade_de_features_descartadas = len(features) - len(top_51_features)
print("Quantidade de features selecionadas (top 51):", len(top_51_features))
print("Quantidade de features descartadas:", quantidade_de_features_descartadas)

# Mostra qual foi o corte de importância para as features selecionadas
importancia_corte = df_importancias[df_importancias['Feature'] == top_51_features[-1]]['Importancia'].values[0]
print(f"Importância mínima para entrar no top 51: {importancia_corte:.6f}")
display(df_importancias.tail(quantidade_de_features_descartadas + 5))

print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))


Iniciando o treinamento do Random Forest para extração de features (MDI)...
Treinamento RF concluído! Tempo extração: 32.9421 segundos.

Quantidade total de features: 78
Quantidade de features selecionadas (top 51): 51
Quantidade de features descartadas: 27
Importância mínima para entrar no top 51: 0.005018


,Feature,Importancia
38,fwd_iat_min,6.755297e-03
54,flag_SYN,6.706613e-03
12,fwd_pkt_per_s,5.392410e-03
13,bwd_pkt_per_s,5.330714e-03
51,idle_min,5.017751e-03
55,flag_fin,4.479901e-03
6,pkt_len_min,4.167894e-03
11,pkt_per_s,3.670549e-03
50,idle_max,3.365798e-03
56,flag_rst,3.119371e-03



Top 10 features mais importantes:


,Feature,Importancia
0,src_port,0.041654
1,dst_port,0.039278
25,bwd_pkt_len_max,0.039001
3,flow_duration,0.034381
27,bwd_pkt_len_mean,0.033061
5,pkt_len_max,0.032568
16,fwd_pkt_len_max,0.030279
4,down_up_ratio,0.027519
20,fwd_pkt_hdr_len_tot,0.027230
19,fwd_pkt_len_std,0.026624


In [17]:
# 5. Reduzindo a dimensionalidade dos conjuntos de treino e teste
X_treino_reduzido = X_treino[top_51_features]
X_teste_reduzido = X_teste[top_51_features]

print(f"Novas dimensões - Treino: {X_treino_reduzido.shape} | Teste: {X_teste_reduzido.shape}")

# Salvando os datasets reduzidos em CSV para uso posterior
X_treino_reduzido.to_csv(caminho_pasta_tratado + nome_dados_treinamento_reduzidos, index=False)
print(f"Dataset de treino reduzido salvo em: {caminho_pasta_tratado + nome_dados_treinamento_reduzidos}")

X_teste_reduzido.to_csv(caminho_pasta_tratado + nome_dados_teste_reduzidos, index=False)
print(f"Dataset de teste reduzido salvo em: {caminho_pasta_tratado + nome_dados_teste_reduzidos}")


Novas dimensões - Treino: (1286248, 51) | Teste: (551250, 51)
Dataset de treino reduzido salvo em: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos_balanceados.csv
Dataset de teste reduzido salvo em: ../../dataset tratado/lycos-cicids2017/Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos_balanceados.csv


In [18]:
# 6. Convertendo as Labels de Texto para Números
print("\nCodificando as labels para a Rede Neural...")
encoder = LabelEncoder()
y_treino_num = encoder.fit_transform(y_treino)
y_teste_num = encoder.transform(y_teste)
num_classes = len(encoder.classes_)


Codificando as labels para a Rede Neural...


In [19]:
# 7. Construindo a Arquitetura da Rede Neural Profunda (DNN) com a entrada reduzida
dnn_model = Sequential([
    keras.layers.Input(shape=(X_treino_reduzido.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

dnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

In [20]:
# 8. O Treinamento (Medindo o Custo Computacional)
print("\nIniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida...")
inicio_treino_dnn = time.time()

history = dnn_model.fit(X_treino_reduzido, y_treino_num, epochs=10, batch_size=256, validation_split=0.1)

fim_treino_dnn = time.time()
tempo_treinamento_dnn = fim_treino_dnn - inicio_treino_dnn
print(f"\nTreinamento DNN concluído! Tempo total: {tempo_treinamento_dnn:.4f} segundos.")


Iniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida...
Epoch 1/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.9824 - loss: 0.0756 - val_accuracy: 0.9950 - val_loss: 0.0186
Epoch 2/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9952 - loss: 0.0175 - val_accuracy: 0.9963 - val_loss: 0.0122
Epoch 3/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9963 - loss: 0.0126 - val_accuracy: 0.9969 - val_loss: 0.0096
Epoch 4/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9966 - loss: 0.0109 - val_accuracy: 0.9969 - val_loss: 0.0087
Epoch 5/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9968 - loss: 0.0099 - val_accuracy: 0.9971 - val_loss: 0.0085
Epoch 6/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9970 - loss: 0.0092 - val_accuracy: 0.9973 - val_loss: 0.0080
Epoch 7/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9972 - loss: 0.0086 - val_accuracy: 0.9975 - val_loss: 0.0075
Ep

In [21]:
# 9. Classificação dos Dados de Teste
print("\nIniciando a classificação dos dados de teste...")
inicio_teste_dnn = time.time()

previsoes_probabilidades = dnn_model.predict(X_teste_reduzido)
previsoes_dnn_num = np.argmax(previsoes_probabilidades, axis=1)

fim_teste_dnn = time.time()
tempo_teste_dnn = fim_teste_dnn - inicio_teste_dnn
print(f"Classificação concluída! Tempo de previsão: {tempo_teste_dnn:.4f} segundos.")

previsoes_dnn_labels = encoder.inverse_transform(previsoes_dnn_num)

# 10. Avaliação dos Resultados
print("\n=== MATRIZ DE CONFUSÃO (DNN - Reduzida) ===")
labels = sorted(np.unique(np.concatenate([y_teste.values, previsoes_dnn_labels])))
cm = confusion_matrix(y_teste, previsoes_dnn_labels, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{lbl}" for lbl in labels],
    columns=[f"Pred_{lbl}" for lbl in labels]
)

def color_confusion_matrix(data):
    styles = pd.DataFrame("", index=data.index, columns=data.columns)
    for i in range(min(data.shape[0], data.shape[1])):
        styles.iat[i, i] = "background-color: #c6efce; color: #006100; font-weight: bold;"
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if i != j and data.iat[i, j] != 0:
                styles.iat[i, j] = "background-color: #ffc7ce; color: #9c0006; font-weight: bold;"
    return styles

display(cm_df.style.apply(color_confusion_matrix, axis=None).format("{:.0f}"))

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_teste, previsoes_dnn_labels, zero_division=0))


Iniciando a classificação dos dados de teste...
17227/17227 ━━━━━━━━━━━━━━━━━━━━ 8s 459us/step
Classificação concluída! Tempo de previsão: 11.1580 segundos.

=== MATRIZ DE CONFUSÃO (DNN - Reduzida) ===


,Pred_benign,Pred_bot,Pred_ddos,Pred_dos_goldeneye,Pred_dos_hulk,Pred_dos_slowhttptest,Pred_dos_slowloris,Pred_ftp_patator,Pred_heartbleed,Pred_portscan,Pred_ssh_patator,Pred_webattack_bruteforce,Pred_webattack_sql_injection,Pred_webattack_xss
Real_benign,418246,5,1,223,2,138,11,1,0,11,1,0,0,0
Real_bot,0,219,0,0,0,0,0,0,0,0,0,0,0,0
Real_ddos,9,0,28587,0,0,0,0,0,0,0,0,0,0,0
Real_dos_goldeneye,34,0,0,2060,1,1,0,0,0,0,0,0,0,0
Real_dos_hulk,10,0,0,0,47931,1,0,0,0,0,0,0,0,0
Real_dos_slowhttptest,14,0,0,1,0,1344,64,0,0,0,0,0,0,0
Real_dos_slowloris,36,0,0,0,0,10,1729,0,0,0,0,0,0,0
Real_ftp_patator,3,0,0,0,0,0,0,1178,0,0,0,0,0,0
Real_heartbleed,1,0,0,0,0,0,0,0,3,0,0,0,0,0
Real_portscan,45,0,0,0,0,0,0,0,0,47862,0,0,0,0



=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===
                         precision    recall  f1-score   support

                 benign       1.00      1.00      1.00    418639
                    bot       0.98      1.00      0.99       219
                   ddos       1.00      1.00      1.00     28596
          dos_goldeneye       0.90      0.98      0.94      2096
               dos_hulk       1.00      1.00      1.00     47942
       dos_slowhttptest       0.90      0.94      0.92      1423
          dos_slowloris       0.96      0.97      0.97      1775
            ftp_patator       1.00      1.00      1.00      1181
             heartbleed       1.00      0.75      0.86         4
               portscan       1.00      1.00      1.00     47907
            ssh_patator       1.00      0.98      0.99       875
   webattack_bruteforce       1.00      0.06      0.11       397
webattack_sql_injection       0.00      0.00      0.00         3
          webattack_xss      

In [22]:
# 7. Exportacao das tabelas em formato LaTeX
from IPython.display import Markdown, display

CLASS_ALIASES_LATEX = {'BENIGN': 'BENIGN', 'Bot': 'Bot', 'DDoS': 'DDoS', 'DoS GoldenEye': 'DoS GE', 'DoS Hulk': 'Hulk', 'DoS Slowhttptest': 'Slowhttp', 'DoS slowloris': 'Slowloris', 'FTP-Patator': 'FTP', 'Heartbleed': 'Heart', 'Infiltration': 'Infil', 'PortScan': 'PortScan', 'SSH-Patator': 'SSH', 'Web Attack - Brute Force': 'Brute', 'Web Attack - Sql Injection': 'SQL', 'Web Attack - XSS': 'XSS', 'Desconhecido': 'Desconh'}


def escape_latex(value):
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def latex_class_name(label, short=False):
    if short:
        return escape_latex(CLASS_ALIASES_LATEX.get(label, label))
    return escape_latex(label)


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return rf"\ok{{{value}}}"
    if value != 0:
        return rf"\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [latex_class_name(label, short=True) for label in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((rf"Real\_{latex_class_name(real_label, short=True)}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \scriptsize",
        r"    \resizebox{\linewidth}{!}{",
        rf"        \begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        r"            \hline",
        format_row("", headers),
        r"            \hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        r"            \hline",
        r"        \end{tabular}",
        r"    }",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values,
        y_pred_values,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            latex_class_name(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        [r"\textbf{Acur?cia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        [r"\textbf{M?dia Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        [r"\textbf{M?dia Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precis?o", "Revoca??o", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + r" \\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \small",
        r"    \begin{tabular}{lrrrr}",
        r"        \hline",
        format_row(headers),
        r"        \hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        r"        \hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        r"        \hline",
        r"    \end{tabular}",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


labels_latex = labels
y_true_latex = y_teste
y_pred_latex = previsoes_dnn_labels

latex_matriz_confusao = make_latex_confusion_matrix(
    cm,
    labels_latex,
    "Matriz de Confus?o - Random Forest",
    "table:rf_completo_mc",
)
latex_relatorio_metricas = make_latex_metrics_report(
    y_true_latex,
    y_pred_latex,
    labels_latex,
    "Relat?rio de M?tricas do Modelo Random Forest",
    "table:rf_completo_metricas",
)

print(latex_matriz_confusao)
print("\n")
print(latex_relatorio_metricas)

\begin{table}[H]
    \centering
    \scriptsize
    \resizebox{\linewidth}{!}{
        \begin{tabular}{l|rrrrrrrrrrrrrr}
            \hline
                                            & benign      & bot      & ddos       & dos\_goldeneye & dos\_hulk  & dos\_slowhttptest & dos\_slowloris & ftp\_patator & heartbleed & portscan   & ssh\_patator & webattack\_bruteforce & webattack\_sql\_injection & webattack\_xss \\
            \hline
            Real\_benign                    & \ok{418246} & \err{5}  & \err{1}    & \err{223}      & \err{2}    & \err{138}         & \err{11}       & \err{1}      & 0          & \err{11}   & \err{1}      & 0                     & 0                         & 0              \\
            Real\_bot                       & 0           & \ok{219} & 0          & 0              & 0          & 0                 & 0              & 0            & 0          & 0          & 0            & 0                     & 0                         & 0              \\
          